In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

def count_per_graph(endpoint_url: str, query: str) -> pd.DataFrame:
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(query)
    results = sparql.query().convert()

    rows = []
    for result in results["results"]["bindings"]:
        graph = result["graph"]["value"]
        count = int(result["count"]["value"])
        rows.append({"graph": graph, "count": count})
    
    df = pd.DataFrame(rows)
    return df


In [4]:
entity_queries = {
    "AOPs": "aopo:AdverseOutcomePathway",
    "KEs": "aopo:KeyEvent",
    "KERs": "aopo:KeyEventRelationship",
    "Stressors": "nci:C54571"
}

# Initialize empty list for partial dataframes
df_list = []
endpoint = "http://localhost:8890/sparql/"
for label, rdf_type in entity_queries.items():
    query = f"""
    SELECT ?graph (COUNT(?s) AS ?count)
    WHERE {{
      GRAPH ?graph {{
        ?s a {rdf_type} .
      }}
      FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
    }}
    GROUP BY ?graph
    ORDER BY ?graph
    """
    df_part = count_per_graph(endpoint, query)
    df_part["version"] = df_part["graph"].str.extract(r"(\d{4}-\d{2})") 
    df_part = df_part[["version", "count"]].rename(columns={"count": label})
    df_list.append(df_part)

# Merge all on version
from functools import reduce
df_all = reduce(lambda left, right: pd.merge(left, right, on="version", how="outer"), df_list)
df_all = df_all.sort_values("version").reset_index(drop=True)

df_all



,version,AOPs,KEs,KERs,Stressors
0,2018-04,219,911,1056,306
1,2018-07,227,936,1085,332
2,2018-10,230,930,1077,332
3,2019-01,236,961,1115,345
4,2019-04,246,980,1144,380
5,2019-07,261,1022,1199,430
6,2019-10,267,1043,1220,434
7,2020-01,273,1060,1239,444
8,2020-04,280,1080,1259,454
9,2020-07,304,1111,1318,477


In [6]:
%pip install plotly
%pip install -U nbformat notebook ipywidgets

import sys
print(sys.executable)


import plotly.express as px

# Plotly line plot with hover
fig = px.line(
    df_all,
    x="version",
    y="AOPs",
    markers=True,
    title="Number of AOPs per AOP-Wiki RDF Version",
    labels={"version": "Version (YYYY-MM)", "AOPs": "Number of AOPs"},
    hover_data={"version": True, "AOPs": True}
)

fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white",
    hovermode="x unified",
    title_font_size=18,
    font=dict(size=14),
    height=500
)

fig.show()


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
/home/marvin/Desktop/.conda/bin/python


In [7]:
import plotly.express as px

fig = px.line(
    df_all,
    x="version",
    y=["AOPs", "KEs", "KERs", "Stressors"],
    markers=True,
    labels={"value": "Count", "variable": "Entity"},
    title="AOP-Wiki Entity Counts Over Time"
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500, hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_all["version"],
        ticktext=df_all["version"]
    ))

fig.show()


In [8]:
# Assuming your DataFrame is called df and sorted by version
df_diff = df_all.copy()

# Compute difference per column (AOPs, KEs, etc.)
for col in ["AOPs", "KEs", "KERs", "Stressors"]:
    df_diff[f"{col}_Δ"] = df_all[col].diff().fillna(0).astype(int)

df_diff


,version,AOPs,KEs,KERs,Stressors,AOPs_Δ,KEs_Δ,KERs_Δ,Stressors_Δ
0,2018-04,219,911,1056,306,0,0,0,0
1,2018-07,227,936,1085,332,8,25,29,26
2,2018-10,230,930,1077,332,3,-6,-8,0
3,2019-01,236,961,1115,345,6,31,38,13
4,2019-04,246,980,1144,380,10,19,29,35
5,2019-07,261,1022,1199,430,15,42,55,50
6,2019-10,267,1043,1220,434,6,21,21,4
7,2020-01,273,1060,1239,444,6,17,19,10
8,2020-04,280,1080,1259,454,7,20,20,10
9,2020-07,304,1111,1318,477,24,31,59,23


In [9]:
import plotly.express as px

# Melt the dataframe into long format for Plotly
df_melted = df_diff.melt(
    id_vars="version",
    value_vars=["AOPs_Δ", "KEs_Δ", "KERs_Δ", "Stressors_Δ"],
    var_name="Entity",
    value_name="Δ Count"
)

# Clean up entity labels
df_melted["Entity"] = df_melted["Entity"].str.replace("_Δ", "")

# Create line plot
fig = px.line(
    df_melted,
    x="version",
    y="Δ Count",
    color="Entity",
    markers=True,
    title="Change in AOP-Wiki Entities Between Versions",
    labels={"version": "Version (YYYY-MM)", "Δ Count": "Entity Increase"},
)

# Set all x-ticks to all available versions
fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500,
    hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_diff["version"],
        ticktext=df_diff["version"]
    )
)

fig.show()


In [10]:
def run_sparql_query(query: str) -> pd.DataFrame:
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(query)
    results = sparql.query().convert()
    return results["results"]["bindings"]

endpoint_url = "http://localhost:8890/sparql/"

In [11]:


def extract_counts(results, var_name):
    return pd.DataFrame([{
        "version": r["graph"]["value"].split("/")[-1],
        var_name: int(r[var_name]["value"])
    } for r in results])

# --- AOP count ---
query_aops = """
SELECT ?graph (COUNT(?aop) AS ?aop_count)
WHERE {
  GRAPH ?graph {
    ?aop a aopo:AdverseOutcomePathway .
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""
df_aops = extract_counts(run_sparql_query(query_aops), "aop_count")

# --- KE count ---
query_kes = """
SELECT ?graph (COUNT(?ke) AS ?ke_count)
WHERE {
  GRAPH ?graph {
    ?aop a aopo:AdverseOutcomePathway ;
         aopo:has_key_event ?ke .
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""
df_kes = extract_counts(run_sparql_query(query_kes), "ke_count")

# --- KER count ---
query_kers = """
SELECT ?graph (COUNT(?ker) AS ?ker_count)
WHERE {
  GRAPH ?graph {
    ?aop a aopo:AdverseOutcomePathway ;
         aopo:has_key_event_relationship ?ker .
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""
df_kers = extract_counts(run_sparql_query(query_kers), "ker_count")

# --- Merge all and compute averages ---
df_all = df_aops.merge(df_kes, on="version").merge(df_kers, on="version")
df_all["avg_KEs_per_AOP"] = df_all["ke_count"] / df_all["aop_count"]
df_all["avg_KERs_per_AOP"] = df_all["ker_count"] / df_all["aop_count"]

# --- Prepare for plotting ---
df_melted = df_all[["version", "avg_KEs_per_AOP", "avg_KERs_per_AOP"]].melt(
    id_vars="version", var_name="Metric", value_name="Average")

df_melted["Metric"] = df_melted["Metric"].replace({
    "avg_KEs_per_AOP": "Average KEs per AOP",
    "avg_KERs_per_AOP": "Average KERs per AOP"
})

# --- Plot ---
fig = px.line(df_melted.sort_values("version"), x="version", y="Average",
              color="Metric", markers=True,
              title="Average KEs and KERs per AOP over Time")

fig.update_layout(template="plotly_white", xaxis_tickangle=-45, height=500, hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_melted["version"],
        ticktext=df_melted["version"]
    ))
fig.show()

In [12]:
# --- KE Reuse Histogram (for one version, e.g., latest) ---

version_to_check = df_melted["version"].max()
query_ke_reuse = f"""
SELECT ?ke (COUNT(DISTINCT ?aop) AS ?aop_count)
WHERE {{
  GRAPH <http://aopwiki.org/graph/{version_to_check}> {{
    ?aop a aopo:AdverseOutcomePathway ;
         aopo:has_key_event ?ke .
  }}
}}
GROUP BY ?ke
"""

results = run_sparql_query(query_ke_reuse)
df_ke_reuse = pd.DataFrame([{
    "ke": r["ke"]["value"],
    "aop_count": int(r["aop_count"]["value"])
} for r in results])

# Histogram plot
fig = px.histogram(df_ke_reuse, x="aop_count", nbins=20,
                   title=f"Distribution of KE Reuse (Version {version_to_check})",
                   labels={"aop_count": "Number of AOPs per KE"})
fig.show()

# Show top 10 most reused KEs
df_ke_reuse.sort_values("aop_count", ascending=False).head(10).reset_index(drop=True)

#DO THIS PER BIO LEVEL

,ke,aop_count
0,https://identifiers.org/aop.events/360,57
1,https://identifiers.org/aop.events/1115,48
2,https://identifiers.org/aop.events/1392,47
3,https://identifiers.org/aop.events/177,33
4,https://identifiers.org/aop.events/351,26
5,https://identifiers.org/aop.events/281,25
6,https://identifiers.org/aop.events/55,21
7,https://identifiers.org/aop.events/18,21
8,https://identifiers.org/aop.events/341,21
9,https://identifiers.org/aop.events/563,21


In [13]:
# --- Network Density Plot ---

query_density = """
SELECT ?graph (COUNT(DISTINCT ?ke) AS ?nodes) (COUNT(?ker) AS ?edges)
WHERE {
  GRAPH ?graph {
    ?aop a aopo:AdverseOutcomePathway ;
         aopo:has_key_event ?ke ;
         aopo:has_key_event_relationship ?ker .
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""

results = run_sparql_query(query_density)
df_density = pd.DataFrame([{
    "version": r["graph"]["value"].split("/")[-1],
    "nodes": int(r["nodes"]["value"]),
    "edges": int(r["edges"]["value"])
} for r in results])
df_density["density"] = 2 * df_density["edges"] / (df_density["nodes"] * (df_density["nodes"] - 1))

fig = px.line(df_density.sort_values("version"), x="version", y="density", 
               title="Network Density of KE-KER Graph", markers=True,
               labels={"density": "Graph Density"})
fig.update_layout(template="plotly_white", xaxis_tickangle=-45, height=500, hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_melted["version"],
        ticktext=df_melted["version"]
    ))
fig.show()


In [14]:
# SPARQL query to count KE Components by type per graph
def run_sparql_query(query: str) -> list:
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(query)
    results = sparql.query().convert()
    return results["results"]["bindings"]

# --- Query: Number of KEs with BiologicalProcess Components over Time ---

query_processes = """
SELECT ?graph (COUNT(DISTINCT ?process) AS ?process_count)
WHERE {
  GRAPH ?graph {
    ?ke a aopo:KeyEvent ;
        aopo:hasBiologicalEvent ?bioevent .
    OPTIONAL { ?bioevent aopo:hasProcess ?process . }
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""

results_processes = run_sparql_query(query_processes)

df_processes = pd.DataFrame([{
    "version": r["graph"]["value"].split("/")[-1],
    "process_count": int(r["process_count"]["value"])
} for r in results_processes]).sort_values("version")

# --- Line Plot: Biological Process Component Use Over Time ---
fig = px.line(df_processes, x="version", y="process_count",
               title="KEs Annotated with Biological Processes Over Time",
               labels={"process_count": "KEs with Process Annotation"},
               markers=True)

fig.update_layout(template="plotly_white", xaxis_tickangle=-45, height=500, hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_melted["version"],
        ticktext=df_melted["version"]
    ))
fig.show()



In [15]:
# --- Query: KE Process Annotation by Ontology (GO, MP, NBO, ...) ---

query_ontologies = """
SELECT ?graph ?ontology (COUNT(DISTINCT ?process) AS ?count)
WHERE {
  GRAPH ?graph {
    ?ke a aopo:KeyEvent ;
        aopo:hasBiologicalEvent ?bioevent .
    ?bioevent aopo:hasProcess ?process .

    BIND(
      IF(STRSTARTS(STR(?process), "http://purl.obolibrary.org/obo/GO_"), "GO",
      IF(STRSTARTS(STR(?process), "http://purl.obolibrary.org/obo/MP_"), "MP",
      IF(STRSTARTS(STR(?process), "http://purl.obolibrary.org/obo/NBO_"), "NBO",
      IF(STRSTARTS(STR(?process), "http://purl.obolibrary.org/obo/MI_"), "MI",
      IF(STRSTARTS(STR(?process), "http://purl.obolibrary.org/obo/VT_"), "VT",
      IF(STRSTARTS(STR(?process), "http://purl.org/commons/record/mesh/"), "MESH",
      IF(STRSTARTS(STR(?process), "http://purl.obolibrary.org/obo/HP_"), "HP", "OTHER"))))))) AS ?ontology)
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph ?ontology
ORDER BY ?graph ?ontology
"""

results_ont = run_sparql_query(query_ontologies)

df_ont = pd.DataFrame([{
    "version": r["graph"]["value"].split("/")[-1],
    "ontology": r["ontology"]["value"],
    "count": int(r["count"]["value"])
} for r in results_ont if "ontology" in r])

# --- Bar Chart: KE Process Annotations by Ontology Over Time ---
fig = px.bar(df_ont.sort_values("version"),
              x="version", y="count", color="ontology", barmode="group",
              title="KEs Annotated with Biological Processes by Ontology",
              labels={"count": "Annotated KEs", "ontology": "Ontology"})

fig.update_layout(template="plotly_white", xaxis_tickangle=-45, height=500, hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_melted["version"],
        ticktext=df_melted["version"]
    ))
fig.show()


In [18]:
# --- Query: Count KE Components by Type ---
query_components = """
SELECT ?graph 
       (COUNT( ?process) AS ?process_count)
       (COUNT( ?object) AS ?object_count)
       (COUNT( ?action) AS ?action_count)
WHERE {
  GRAPH ?graph {
    ?ke a aopo:KeyEvent ;
        aopo:hasBiologicalEvent ?bioevent .
    OPTIONAL { ?bioevent aopo:hasProcess ?process . }
    OPTIONAL { ?bioevent aopo:hasObject ?object . }
    OPTIONAL { ?bioevent aopo:hasAction ?action . }
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""

results_components = run_sparql_query(query_components)

df_components = pd.DataFrame([{
    "Version": r["graph"]["value"].split("/")[-1],
    "Process": int(r["process_count"]["value"]),
    "Object": int(r["object_count"]["value"]),
    "Action": int(r["action_count"]["value"])
} for r in results_components]).sort_values("Version")

df_melted = df_components.melt(
    id_vars=["Version"], 
    value_vars=["Process", "Object", "Action"],
    var_name="Component", value_name="Count"
)

fig1 = px.line(
    df_melted, 
    x="Version", 
    y="Count", 
    color="Component",
    title="Total number of KE Component annotations over time",
    markers=True
)
fig1.update_layout(title_x=0.5)

fig1.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500,
    hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_components["Version"],
        ticktext=df_components["Version"]
    )
)
fig1.show()

df_diff = df_components.copy()
df_diff[["Process", "Object", "Action"]] = \
    df_diff[["Process", "Object", "Action"]].diff()

df_melted_diff = df_diff.melt(
    id_vars=["Version"], 
    value_vars=["Process", "Object", "Action"],
    var_name="Component", value_name="Change"
)

fig2 = px.bar(
    df_melted_diff, 
    x="Version", 
    y="Change", 
    color="Component",
    title="Quarterly change in total KE Component annotations",
    barmode="group"
)
fig2.update_layout(title_x=0.5)

fig2.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500,
    xaxis=dict(
        tickmode='array',
        tickvals=df_components["Version"],
        ticktext=df_components["Version"]
    )
)
fig2.show()

%pip install -U kaleido

# Save line plot as PNG (high detail)
fig1.write_image("KE_components_over_time.png", format="png", scale=4, width=800, height=500)

# Save bar plot as PNG (high detail)
fig2.write_image("KE_components_quarterly_change.png", format="png", scale=4, width=800, height=500)


Note: you may need to restart the kernel to use updated packages.


: 

: 

In [ ]:
# --- Query: Count KE Components by Type ---
query_components = """
SELECT ?graph 
       (COUNT(DISTINCT ?process) AS ?process_count)
       (COUNT(DISTINCT ?object) AS ?object_count)
       (COUNT(DISTINCT ?action) AS ?action_count)
WHERE {
  GRAPH ?graph {
    ?ke a aopo:KeyEvent ;
        aopo:hasBiologicalEvent ?bioevent .
    OPTIONAL { ?bioevent aopo:hasProcess ?process . }
    OPTIONAL { ?bioevent aopo:hasObject ?object . }
    OPTIONAL { ?bioevent aopo:hasAction ?action . }
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
"""

results_components = run_sparql_query(query_components)

df_components = pd.DataFrame([{
    "Version": r["graph"]["value"].split("/")[-1],
    "process_count": int(r["process_count"]["value"]),
    "object_count": int(r["object_count"]["value"]),
    "action_count": int(r["action_count"]["value"])
} for r in results_components]).sort_values("Version")

df_melted = df_components.melt(
    id_vars=["Version"], 
    value_vars=["process_count", "object_count", "action_count"],
    var_name="Component", value_name="Count"
)

fig1 = px.line(
    df_melted, 
    x="Version", 
    y="Count", 
    color="Component",
    title="Number of unique KE Component annotations over time",
    markers=True
)
fig1.update_layout(title_x=0.5)
fig1.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500,
    hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_components["Version"],
        ticktext=df_components["Version"]
    )
)
fig1.show()

df_diff = df_components.copy()
df_diff[["process_change", "object_change", "action_change"]] = \
    df_diff[["process_count", "object_count", "action_count"]].diff()

df_melted_diff = df_diff.melt(
    id_vars=["Version"], 
    value_vars=["process_change", "object_change", "action_change"],
    var_name="Component", value_name="Change"
)

fig2 = px.bar(
    df_melted_diff, 
    x="Version", 
    y="Change", 
    color="Component",
    title="Monthly change in unique KE Component annotations",
    barmode="group"
)
fig2.update_layout(title_x=0.5)
fig2.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    height=500,
    xaxis=dict(
        tickmode='array',
        tickvals=df_components["Version"],
        ticktext=df_components["Version"]
    )
)
fig2.show()

: 

In [ ]:
# --- Query: Number of Unique AOP Creators per Version ---
query_authors = """
SELECT ?graph (COUNT(DISTINCT ?c) AS ?author_count)
WHERE {
  GRAPH ?graph {
    ?aop a aopo:AdverseOutcomePathway ;
         dc:creator ?c .
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
GROUP BY ?graph
ORDER BY ?graph
"""

# Run query and convert results
results_authors = run_sparql_query(query_authors)

df_authors = pd.DataFrame([{
    "version": r["graph"]["value"].split("/")[-1],
    "author_count": int(r["author_count"]["value"])
} for r in results_authors]).sort_values("version")

# --- Plot ---
fig = px.line(df_authors, x="version", y="author_count", markers=True,
              title="Number of Unique AOP Authors per Version",
              labels={"author_count": "Unique Authors", "version": "AOP-Wiki Version"})

fig.update_layout(template="plotly_white", xaxis_tickangle=-45, height=500, hovermode="x unified",
    xaxis=dict(
        tickmode='array',
        tickvals=df_melted["version"],
        ticktext=df_melted["version"]
    ))
fig.show()

: 

: 

: 

: 

In [ ]:
# --- Query AOP creation and modification dates ---
query_lifetime = """
SELECT ?graph ?aop ?created ?modified
WHERE {
  GRAPH ?graph {
    ?aop a aopo:AdverseOutcomePathway ;
         dcterms:created ?created ;
         dcterms:modified ?modified .
  }
  FILTER(STRSTARTS(STR(?graph), "http://aopwiki.org/graph/"))
}
"""

# --- Run query and format result ---
results_lifetime = run_sparql_query(query_lifetime)
df_lifetime = pd.DataFrame([{
    "aop": r["aop"]["value"],
    "version": r["graph"]["value"].split("/")[-1],
    "created": pd.to_datetime(r["created"]["value"]),
    "modified": pd.to_datetime(r["modified"]["value"])
} for r in results_lifetime])

# --- Derived fields ---
df_lifetime["lifetime_days"] = (df_lifetime["modified"] - df_lifetime["created"]).dt.days
df_lifetime["year_created"] = df_lifetime["created"].dt.year
df_lifetime["year_modified"] = df_lifetime["modified"].dt.year

# --- Deduplicate: One row per unique AOP URI ---

# First version where the AOP was created (keep earliest creation timestamp)
df_created = df_lifetime.sort_values("created").drop_duplicates("aop", keep="first")

# Last version where the AOP was modified (keep latest modification timestamp)
df_modified = df_lifetime.sort_values("modified").drop_duplicates("aop", keep="last")

# --- 1. AOPs created per year (unique) ---
fig1 = px.histogram(df_created, x=df_created["created"].dt.year,
                    title="Number of Unique AOPs Created per Year",
                    labels={"x": "Year of Creation", "y": "AOP Count"},
                    color_discrete_sequence=["#636EFA"])
fig1.update_layout(template="plotly_white", height=400)
fig1.show()

# --- 2. AOPs last modified per year (unique) ---
fig2 = px.histogram(df_modified, x=df_modified["modified"].dt.year,
                    title="Number of Unique AOPs Modified per Year",
                    labels={"x": "Year of Last Modification", "y": "AOP Count"},
                    color_discrete_sequence=["#EF553B"])
fig2.update_layout(template="plotly_white", height=400)
fig2.show()

# --- 3. Scatter plot: Created vs Modified per AOP ---
fig3 = px.scatter(df_lifetime, x="created", y="modified", hover_name="aop",
                  title="Creation vs. Last Modification Dates of AOPs",
                  labels={"created": "Date Created", "modified": "Date Modified"},
                  color_discrete_sequence=["#AB63FA"])
fig3.update_layout(template="plotly_white", height=500)
fig3.show()

: 

: 

: 

: 